# Cardiac Segmentation — Mamba Integration Study
**Full Q1 Paper Pipeline — Google Colab Pro**

This notebook trains all models, runs parameter-matched baselines, evaluates on the test set,
computes EF metrics, and generates publication-ready tables.

## Requirements
- Colab Pro with A100 GPU (V100/T4 work but slower)
- CAMUS dataset in Google Drive at `CAMUS_public/database_nifti/`

## Session Plan (3 sessions)
| Session | What | Time (A100) |
|---------|------|-------------|
| 1 | Base models (9) + Mamba models (9) | ~4-5h |
| 2 | Mamba2 (9) + VMamba (9) | ~6-8h |
| 3 | Parameter-matched baselines (9) + Evaluation + EF + Benchmarks | ~3-4h |

---
## 1. Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install PyTorch with CUDA 12.1
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# Install mamba-ssm + causal-conv1d (CUDA kernels — takes 2-5 min)
!pip uninstall -y mamba-ssm causal-conv1d 2>/dev/null
!pip install causal-conv1d mamba-ssm --no-build-isolation -q

In [ ]:
# Clone repo (always pull latest)
import os
if os.path.exists('/content/mamba'):
    !cd /content/mamba && git pull
else:
    !git clone https://github.com/Seraf00/mamba.git /content/mamba
%cd /content/mamba

In [ ]:
# Install remaining dependencies
!pip install uv -q
!uv pip install -r requirements.txt --quiet

In [ ]:
# Suppress noisy warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TRITON_F32_DEFAULT'] = 'ieee'
import warnings
warnings.filterwarnings('ignore')

## 2. Copy CAMUS Dataset

In [ ]:
source_path = '/content/drive/MyDrive/CAMUS_public/database_nifti/'
destination_path = '/content/mamba/data/CAMUS'
os.makedirs(destination_path, exist_ok=True)

if len(os.listdir(destination_path)) < 100:
    print('Copying CAMUS dataset... (5-10 min)')
    !rsync -ah --info=progress2 "{source_path}" "{destination_path}"
    print('Done!')
else:
    print(f'CAMUS dataset already present ({len(os.listdir(destination_path))} items)')

## 3. Verify Setup

In [ ]:
!python scripts/check_mamba_setup.py

In [ ]:
# TensorBoard for live monitoring
%load_ext tensorboard

## 4. Drive Sync Helper
Colab sessions die — always sync after training.

In [ ]:
DRIVE_RESULTS = '/content/drive/MyDrive/Paper1/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

def sync_to_drive(local_dir='/content/results'):
    """Sync all results to Google Drive."""
    !rsync -ah --info=progress2 "{local_dir}/" "{DRIVE_RESULTS}/"
    print(f'Synced to {DRIVE_RESULTS}')

---
# SESSION 1: Base + Mamba Models

## 5. Train Base Models (9 models, ~1-2h on A100)

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 16 \
    --early_stopping 20 \
    --num_workers 4 \
    --base_only \
    --mixed_precision \
    --exp_name base_models \
    --output_dir /content/results/

sync_to_drive()

## 6. Train Mamba-Enhanced Models (9 models, ~2-3h on A100)
Uses CUDA-optimized selective scan (~1-2s/batch).

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba \
    --mixed_precision \
    --exp_name mamba_models \
    --output_dir /content/results/

sync_to_drive()

---
# SESSION 2: Mamba2 + VMamba Models

## 7. Train Mamba2-Enhanced Models (9 models, ~3-4h)
Uses SSD algorithm with Triton kernels.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba2 \
    --mixed_precision \
    --exp_name mamba2_models \
    --output_dir /content/results/

sync_to_drive()

## 8. Train VMamba-Enhanced Models (9 models, ~4-6h)
Uses 4-directional cross-scan. AMP dtype fix ensures CUDA kernel works.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 4 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants vmamba \
    --mixed_precision \
    --exp_name vmamba_models \
    --output_dir /content/results/

sync_to_drive()

---
# SESSION 3: Parameter-Matched Baselines + Evaluation

## 9. Compute Parameter-Matched Base Features
Finds wider base models that match Mamba param counts for fair comparison.
A reviewer will ask: *Is the improvement from Mamba or from more parameters?*

In [ ]:
!python scripts/param_match.py --output_json /content/results/param_config.json

## 10. Train Parameter-Matched Wider Base Models (~2-3h)

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --base_only \
    --param_matched \
    --param_config /content/results/param_config.json \
    --mixed_precision \
    --exp_name param_matched \
    --output_dir /content/results/

sync_to_drive()

## 11. Efficiency Benchmarks
Measures FLOPs, latency, memory, and parameter counts for all models.

In [ ]:
!python scripts/benchmark.py \
    --all \
    --mamba_types mamba mamba2 vmamba \
    --output /content/results/benchmark_efficiency.csv

sync_to_drive()

## 12. Evaluate All Models on Test Set
Computes Dice, IoU, HD95, ASD + biplane EF with pixel spacing.
Generates LaTeX tables and statistical tests.

In [ ]:
import glob

# Find all experiment directories that contain trained models
exp_dirs = [
    '/content/results/base_models',
    '/content/results/mamba_models',
    '/content/results/mamba2_models',
    '/content/results/vmamba_models',
    '/content/results/param_matched',
]

for exp_dir in exp_dirs:
    if os.path.exists(exp_dir):
        print(f'\n{"="*70}')
        print(f'Evaluating: {exp_dir}')
        print(f'{"="*70}')
        !python scripts/evaluate_all_models.py \
            --checkpoint_dir "{exp_dir}" \
            --data_dir /content/mamba/data/CAMUS/ \
            --compute_ef \
            --split test \
            --batch_size 8

sync_to_drive()

## 13. TensorBoard (optional — run anytime)

In [ ]:
%tensorboard --logdir /content/results/

## 14. Final Sync to Google Drive

In [ ]:
sync_to_drive()
print('\nAll done! Results are in Google Drive at:', DRIVE_RESULTS)
print('\nGenerated files:')
print('  - results_table.tex    (LaTeX table for paper)')
print('  - evaluation_results.json  (full metrics per model)')
print('  - benchmark_efficiency.csv (FLOPs, latency, memory)')
print('  - summary.csv          (training summary per model)')

---
## Recovery: Resume After Session Timeout
If Colab disconnects mid-training, use `--resume_from` to skip completed models.

In [ ]:
# Example: resume Mamba training from mamba_fpn_mamba
# First, copy previous results from Drive back to Colab
# !rsync -ah "{DRIVE_RESULTS}/mamba_models/" /content/results/mamba_models/

# Then resume from the model that was interrupted
# !python scripts/train_all_models.py \
#     --data_dir /content/mamba/data/CAMUS/ \
#     --epochs 100 --batch_size 8 --early_stopping 20 \
#     --num_workers 4 --mamba_only --mamba_variants mamba \
#     --mixed_precision \
#     --resume_from mamba_fpn_mamba \
#     --exp_name mamba_models \
#     --output_dir /content/results/